In [0]:

# dbutils.widgets.text("table_name", "")
# dbutils.widgets.text("landing_timestamp", "")
# dbutils.widgets.text("merge_keys", "")
# dbutils.widgets.text("load_type", "")

# table_name = dbutils.widgets.get("table_name")
# landing_timestamp = dbutils.widgets.get("landing_timestamp")
# merge_keys = dbutils.widgets.get("merge_keys")
# load_type = dbutils.widgets.get("load_type")


In [0]:
# #read data from bronze to silver only from the bronze timestamp
# bronze_df=spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
# #now we have give insert , updated timestamp for silver
# # from pyspark.sql.functions import current_timestamp, lit
# updated_df=bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("updated_timestamp", current_timestamp())


In [0]:
# #now we will overwrite this data to silver table
# updated_df.write.mode("overwrite").saveAsTable(f"adb_caresync.silver.{table_name}")

In [0]:
%sql
-- select * from adb_caresync.silver.hospitals

In [0]:
# record_count=updated_df.count()
# # dbutils.notebook.exit(str(record_count))

In [0]:
# ============================================================
# STEP 0 - PARAMETERS
# ============================================================

dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 60)
print("     BRONZE TO SILVER  |  NOTEBOOK STARTED")
print("=" * 60)
print("[STEP 0] Parameters Received")
print("-" * 60)
print(f"  {'LANDING TIMESTAMP':<25}: {landing_timestamp}")
print(f"  {'TABLE NAME':<25}: {table_name}")
print(f"  {'MERGE KEYS':<25}: {merge_keys if merge_keys else '(none)'}")
print(f"  {'LOAD TYPE':<25}: {load_type if load_type else '(empty - defaults to FULL)'}")
print("-" * 60)


# ============================================================
# STEP 1 - READ BRONZE SOURCE
# ============================================================

print("\n" + "=" * 60)
print("[STEP 1] Reading Source Bronze Table")
print("=" * 60)

source_table     = f"adb_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

source_query = (
    f"SELECT *\n"
    f"FROM   {source_table}\n"
    f"WHERE  {filter_condition}"
)

print(f"  SOURCE TABLE : {source_table}")
print(f"  FILTER       : {filter_condition}")
print(f"\n[STEP 1 - SOURCE QUERY PREPARED]")
print("-" * 60)
print(source_query)
print("-" * 60)

bronze_df = (
    spark.read
    .table(source_table)
    .filter(filter_condition)
)

source_row_count = bronze_df.count()

print(f"\n[STEP 1 - SOURCE DATA LOADED]")
print(f"  TOTAL SOURCE ROWS : {source_row_count}")
print(f"  COLUMN COUNT      : {len(bronze_df.columns)}")
print(f"  COLUMNS           : {bronze_df.columns}")
print("\n[SOURCE SCHEMA]")
bronze_df.printSchema()
print("[STEP 1 COMPLETE]")


# ============================================================
# STEP 2 - PREPARE METADATA
# ============================================================

from pyspark.sql import functions as F

print("\n" + "=" * 60)
print("[STEP 2] Preparing Load Metadata")
print("=" * 60)

target_table = f"adb_caresync.silver.{table_name}"

normalized_load_type = (load_type or "FULL").strip().upper()

merge_key_list = [
    key.strip()
    for key in (merge_keys or "").split(",")
    if key.strip()
]

print(f"  TARGET TABLE         : {target_table}")
print(f"  RAW LOAD TYPE        : '{load_type}'")
print(f"  NORMALIZED LOAD TYPE : {normalized_load_type}")
print(f"  MERGE KEYS           : {merge_key_list}")


# ============================================================
# VALIDATE LOAD TYPE
# ============================================================

allowed_load_types = ["FULL", "APPEND", "MERGE"]

print(f"\n[STEP 2 - LOAD TYPE VALIDATION]")
print(f"  Allowed  : {allowed_load_types}")
print(f"  Received : '{normalized_load_type}'")

if normalized_load_type not in allowed_load_types:
    raise ValueError(
        f"load_type must be one of: {allowed_load_types}. "
        f"Received: '{load_type}'"
    )

print(f"  Status   : PASSED")


# ============================================================
# VALIDATE MERGE KEYS
# ============================================================

if normalized_load_type == "MERGE":

    print(f"\n[STEP 2 - MERGE KEY VALIDATION]")
    print(f"  Keys to validate  : {merge_key_list}")
    print(f"  Available columns : {bronze_df.columns}")

    if not merge_key_list:
        raise ValueError(
            "merge_keys must be provided when load_type is 'MERGE'."
        )

    missing_merge_keys = [
        key
        for key in merge_key_list
        if key not in bronze_df.columns
    ]

    if missing_merge_keys:
        raise ValueError(
            f"merge_keys not found in source data: "
            f"{missing_merge_keys}"
        )

    print(f"  Status            : ALL KEYS FOUND")


# ============================================================
# ADD AUDIT COLUMNS
# ============================================================

print(f"\n[STEP 2 - ADDING AUDIT COLUMNS]")
print(f"  + insert_timestamp = current_timestamp()")
print(f"  + update_timestamp = current_timestamp()")

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print(f"\n[STEP 2 - UPDATED DATAFRAME]")
print(f"  Column count : {len(updated_df.columns)}")
print(f"  Columns      : {updated_df.columns}")
print("\n[UPDATED SCHEMA]")
updated_df.printSchema()
print("[STEP 2 COMPLETE]")


# ============================================================
# STEP 3 - WRITE TO SILVER
# ============================================================

print("\n" + "=" * 60)
print("[STEP 3] Writing to Target Silver Table")
print("=" * 60)
print(f"  TARGET TABLE : {target_table}")
print(f"  LOAD TYPE    : {normalized_load_type}")
print(f"  ROWS TO WRITE: {source_row_count}")


# ============================================================
# FULL LOAD
# ============================================================

if normalized_load_type == "FULL":

    print(f"\n[STEP 3 - FULL LOAD]")
    print(f"  Strategy     : Overwrite entire target table")
    print(f"  overwriteSchema=True (replaces existing schema if changed)")
    print(f"\n[STEP 3 - WRITE OPERATION]")
    print("-" * 60)
    print(f"  updated_df.write")
    print(f"    .mode('overwrite')")
    print(f"    .option('overwriteSchema', 'true')")
    print(f"    .saveAsTable('{target_table}')")
    print("-" * 60)
    print("  Executing write ...")

    # overwriteSchema=True allows the existing Delta table
    # schema to be replaced by the incoming DataFrame schema.
    updated_df.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    post_write_count = spark.read.table(target_table).count()

    print(f"\n[STEP 3 - POST-WRITE VERIFICATION]")
    print(f"  Rows in {target_table} : {post_write_count}")
    print("[STEP 3 COMPLETE] FULL overwrite finished successfully")


# ============================================================
# APPEND LOAD
# ============================================================

elif normalized_load_type == "APPEND":

    print(f"\n[STEP 3 - APPEND LOAD]")
    print(f"  Strategy : Append new records to target table")

    target_exists = spark.catalog.tableExists(target_table)

    print(f"  TARGET EXISTS : {target_exists}")

    if not target_exists:

        print(f"\n[STEP 3 - APPEND FALLBACK] Target table does not exist - creating.")
        print("-" * 60)
        print(f"  updated_df.write")
        print(f"    .mode('overwrite')")
        print(f"    .saveAsTable('{target_table}')")
        print("-" * 60)
        print("  Executing write ...")

        updated_df.write \
            .mode("overwrite") \
            .saveAsTable(target_table)

    else:

        print(f"\n[STEP 3 - APPEND WRITE OPERATION]")
        print("-" * 60)
        print(f"  updated_df.write")
        print(f"    .mode('append')")
        print(f"    .saveAsTable('{target_table}')")
        print("-" * 60)
        print("  Executing write ...")

        updated_df.write \
            .mode("append") \
            .saveAsTable(target_table)

    post_write_count = spark.read.table(target_table).count()

    print(f"\n[STEP 3 - POST-WRITE VERIFICATION]")
    print(f"  Rows in {target_table} after append : {post_write_count}")
    print("[STEP 3 COMPLETE] APPEND finished successfully")


# ============================================================
# MERGE LOAD
# ============================================================

elif normalized_load_type == "MERGE":

    print(f"\n[STEP 3 - MERGE LOAD]")
    print(f"  Strategy   : Upsert records using merge keys")
    print(f"  Merge Keys : {merge_key_list}")

    target_exists = spark.catalog.tableExists(target_table)

    print(f"  TARGET EXISTS : {target_exists}")


    # --------------------------------------------------------
    # Target doesn't exist
    # --------------------------------------------------------

    if not target_exists:

        print(f"\n[STEP 3 - MERGE FALLBACK] Target table does not exist - creating.")
        print("-" * 60)
        print(f"  updated_df.write")
        print(f"    .mode('overwrite')")
        print(f"    .saveAsTable('{target_table}')")
        print("-" * 60)
        print("  Executing write ...")

        updated_df.write \
            .mode("overwrite") \
            .saveAsTable(target_table)

        post_write_count = spark.read.table(target_table).count()

        print(f"\n[STEP 3 - POST-WRITE VERIFICATION]")
        print(f"  Rows in {target_table} : {post_write_count}")
        print("[STEP 3 COMPLETE] Target table created for initial MERGE load")


    # --------------------------------------------------------
    # Target exists
    # --------------------------------------------------------

    else:

        print(f"\n[STEP 3 - MERGE] Target table exists. Preparing MERGE SQL.")

        source_view = "silver_merge_source"

        updated_df.createOrReplaceTempView(source_view)

        print(f"  Temp view created : '{source_view}'")


        # ----------------------------------------------------
        # Build MERGE condition
        # ----------------------------------------------------

        merge_condition = " AND ".join(
            [
                f"target.`{key}` = source.`{key}`"
                for key in merge_key_list
            ]
        )

        print(f"\n[MERGE CONDITION BUILT]")
        print(f"  ON {merge_condition}")


        # ----------------------------------------------------
        # Build UPDATE clause
        #
        # Do NOT update insert_timestamp for existing records.
        # update_timestamp WILL be updated.
        # ----------------------------------------------------

        update_columns = [
            column
            for column in updated_df.columns
            if column != "insert_timestamp"
        ]

        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in update_columns
            ]
        )

        print(f"\n[UPDATE CLAUSE BUILT]")
        print(f"  Columns updating    ({len(update_columns)}) : {update_columns}")
        print(f"  Excluded (preserved): ['insert_timestamp']")


        # ----------------------------------------------------
        # Build INSERT clause
        # ----------------------------------------------------

        insert_columns = ", ".join(
            [
                f"`{column}`"
                for column in updated_df.columns
            ]
        )

        insert_values = ", ".join(
            [
                f"source.`{column}`"
                for column in updated_df.columns
            ]
        )

        print(f"\n[INSERT CLAUSE BUILT]")
        print(f"  Columns inserting ({len(updated_df.columns)}) : {updated_df.columns}")


        # ----------------------------------------------------
        # Build MERGE SQL
        # ----------------------------------------------------

        merge_sql = f"""
        MERGE INTO {target_table} AS target
        USING {source_view} AS source
        ON {merge_condition}

        WHEN MATCHED THEN
            UPDATE SET {update_assignments}

        WHEN NOT MATCHED THEN
            INSERT ({insert_columns})
            VALUES ({insert_values})
        """

        print(f"\n[STEP 3 - MERGE SQL PREPARED]")
        print("=" * 60)
        print(merge_sql)
        print("=" * 60)

        print("[STEP 3 - EXECUTING MERGE SQL ...]")

        spark.sql(merge_sql)

        post_write_count = spark.read.table(target_table).count()

        print(f"\n[STEP 3 - POST-MERGE VERIFICATION]")
        print(f"  Source rows processed : {source_row_count}")
        print(f"  Total rows in {target_table} after merge : {post_write_count}")
        print("[STEP 3 COMPLETE] MERGE finished successfully")


# ============================================================
# STEP 4 - RECORD COUNT
# ============================================================

print("\n" + "=" * 60)
print("[STEP 4] Final Record Count Summary")
print("=" * 60)

record_count = updated_df.count()

print(f"  Source (bronze) rows processed : {record_count}")
print(f"  Load type used                 : {normalized_load_type}")
print(f"  Target table                   : {target_table}")
print("[STEP 4 COMPLETE]")


# ============================================================
# STEP 5 - EXIT
# ============================================================

print("\n" + "=" * 60)
print("[STEP 5] Notebook Completed Successfully")
print("=" * 60)
print(f"  TABLE NAME   : {table_name}")
print(f"  LOAD TYPE    : {normalized_load_type}")
print(f"  RECORD COUNT : {record_count}")
print(f"  EXIT VALUE   : '{record_count}'")
print("=" * 60)

dbutils.notebook.exit(str(record_count))